In [ ]:
!export PATH="${HOME}/.local/bin:${PATH}" && uv pip uninstall --system jax


In [ ]:
!export PATH="${HOME}/.local/bin:${PATH}" && uv pip install --system tensorflow-tpu=="2.18.0" --find-links https://storage.googleapis.com/libtpu-tf-releases/index.html

In [ ]:
import tensorflow as tf
print('TensorFlow version' + tf.__version__)

In [ ]:
# --- CELL 2 (Run only after resetting session) ---
import tensorflow as tf
from tensorflow import keras

print(f"TF Version: {tf.__version__}")

strategy = None
device_type = "CPU"

# TPU Initialization
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local") # TPU detection
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print("TPU detected")
except Exception as e:
    print(f"Could not initialize TPU: {e}")
    strategy = tf.distribute.get_strategy()  # Fallback to default CPU/GPU
    print("Running on CPU or single GPU") # Fallback to CPU/GPU
print('Number of replicas:', strategy.num_replicas_in_sync)



# Precision + Exec config setup for max throughput
if device_type == "TPU":
    # TPUs are natively optimized for bfloat16
    keras.mixed_precision.set_global_policy("mixed_bfloat16")
    STEPS_PER_EXEC = 32
    JIT = True
else:
    keras.mixed_precision.set_global_policy("float32")
    STEPS_PER_EXEC = 1
    JIT = False

NUM_REPLICAS = strategy.num_replicas_in_sync
print(f"device: {device_type} | replicas: {NUM_REPLICAS} | steps_per_exec: {STEPS_PER_EXEC} | policy: {keras.mixed_precision.global_policy().name}")
JIT_COMPILE = JIT

In [ ]:
from pathlib import Path

# ---- Paths -----------------------------------------------------------------
DATA_ROOT = Path("/kaggle/input/datasets/noahbadoa/plantnet-300k-images")   # change to your Kaggle dataset slug
OUTPUT    = Path("/kaggle/working")

# ---- Training --------------------------------------------------------------
EPOCHS            = 100
PHASE1_EPOCHS     = 3
FINE_TUNE_LAYERS  = 80
PER_REPLICA_BATCH = 128      # global batch = 128 * 8 cores = 1024 on v5e-8
LR_PHASE1         = 1e-3
LR_PHASE2_PEAK    = 5e-4
WARMUP_EPOCHS     = 2
WEIGHT_DECAY      = 1e-4
LABEL_SMOOTHING   = 0.05
GRAD_CLIP_NORM    = 1.0
DROPOUT           = 0.3
IMAGE_SIZE        = 224

# ---- Switches --------------------------------------------------------------
BALANCE   = True      # inverse-frequency class weights for long-tail
QUICK     = False     # True → 5-epoch smoke test
RESUME    = False     # True → load last.keras and continue
STEPS_PER_EXEC = 128  # TPU host-RPC amortization; leave alone

if QUICK:
    PHASE1_EPOCHS = 1
    EPOCHS = 5
    print("QUICK mode → 5 total epochs")

In [ ]:
from collections import Counter
import json

from tensorflow import keras
from tensorflow.keras import layers


def find_split_dir(root, split):
    for c in [root / "plantnet_300K" / "images" / split,
              root / "plantnet_300K" / f"images_{split}",
              root / "images" / split,
              root / f"images_{split}"]:
        if c.is_dir():
            return c
    raise FileNotFoundError(f"{split} split not found under {root}")


train_dir = find_split_dir(DATA_ROOT, "train")
val_dir   = find_split_dir(DATA_ROOT, "val")
print("Train dir:", train_dir)
print("Val dir:  ", val_dir)

GLOBAL_BATCH = PER_REPLICA_BATCH * strategy.num_replicas_in_sync
print("Global batch:", GLOBAL_BATCH)

AUGMENT = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.10),
    layers.RandomBrightness(0.10),
], name="augmentation")


def make_dataset(directory, shuffle, batch_size, augment):
    # 1. Provide the batch_size here so it reads and groups 1024 images immediately
    ds = keras.utils.image_dataset_from_directory(
        directory,
        labels="inferred",
        label_mode="int",
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=batch_size,   # <--- CHANGED: Batch right away
        shuffle=shuffle,         # <--- CHANGED: This handles shuffling internally
        interpolation="bilinear",
    )
    class_names = ds.class_names

    opts = tf.data.Options()
    opts.experimental_deterministic = False
    ds = ds.with_options(opts)

    AUTOTUNE = tf.data.AUTOTUNE
    
    # 2. Map the augmentation over the full BATCH of 1024 images (100x faster)
    if augment is not None:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
        
    # Note: We removed the separate ds.shuffle() and ds.batch() calls here 
    # because image_dataset_from_directory handles it natively now.

    if not shuffle:
        ds = ds.cache()
        
    ds = ds.prefetch(AUTOTUNE)
    return ds, class_names

train_ds, class_names = make_dataset(train_dir, shuffle=True,  batch_size=GLOBAL_BATCH, augment=AUGMENT)
val_ds,   _           = make_dataset(val_dir,   shuffle=False, batch_size=GLOBAL_BATCH, augment=None)
NUM_CLASSES = len(class_names)
print("Classes:", NUM_CLASSES)

OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / "class_index.json").write_text(json.dumps(class_names, indent=2))

# Class weights
class_weight = None
if BALANCE:
    wpath = OUTPUT / "class_weights.json"
    if wpath.exists():
        class_weight = {int(k): float(v) for k, v in json.loads(wpath.read_text()).items()}
    else:
        counts = Counter()
        for idx, name in enumerate(class_names):
            folder = train_dir / name
            counts[idx] = sum(1 for _ in folder.iterdir()) if folder.is_dir() else 0
        n_samples = sum(counts.values())
        class_weight = {idx: (n_samples / (NUM_CLASSES * c)) if c > 0 else 0.0
                        for idx, c in counts.items()}
        wpath.write_text(json.dumps(class_weight, indent=2))
    print(f"Class-weight range: [{min(class_weight.values()):.3f}, {max(class_weight.values()):.3f}]")

steps_per_epoch = tf.data.experimental.cardinality(train_ds).numpy()
if steps_per_epoch <= 0:
    steps_per_epoch = sum(1 for _ in train_dir.rglob("*.jpg")) // GLOBAL_BATCH
print("Steps per epoch:", steps_per_epoch)

In [ ]:
import math
from tensorflow.keras.applications import MobileNetV3Small

# TPU wants bfloat16, not float16
keras.mixed_precision.set_global_policy("mixed_bfloat16")


def build_model(num_classes):
    # ... (backbone setup stays the same) ...

    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), dtype=tf.float32)
    
    # Place augmentation directly inside the model!
    x = AUGMENT(inputs) 
    
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(num_classes, activation="softmax",
                           dtype="float32", name="predictions")(x)
    return keras.Model(inputs, outputs, name="plantnet_classifier")


def loss_fn(y_true, y_pred):
    y_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=tf.shape(y_pred)[-1])
    return keras.losses.categorical_crossentropy(y_oh, y_pred, label_smoothing=LABEL_SMOOTHING)


def compile_model(model, lr_schedule):
    opt = keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=WEIGHT_DECAY,
        global_clipnorm=GRAD_CLIP_NORM,
    )
    model.compile(
        optimizer=opt,
        loss=loss_fn,
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="top1"),
            keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5"),
        ],
        jit_compile=JIT_COMPILE,
        steps_per_execution=STEPS_PER_EXEC,
    )

def cosine_warmup_schedule(peak_lr, total_steps, warmup_steps):
    return keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.0,
        decay_steps=max(1, total_steps - warmup_steps),
        alpha=0.0,
        warmup_target=peak_lr,
        warmup_steps=max(1, warmup_steps),
        name="cosine_warmup",
    )

In [ ]:
BEST_PATH = OUTPUT / "best.keras"
LAST_PATH = OUTPUT / "last.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(str(BEST_PATH), monitor="val_top1", mode="max",
                                    save_best_only=True, verbose=1),
    keras.callbacks.ModelCheckpoint(str(LAST_PATH), save_best_only=False, verbose=0),
    keras.callbacks.CSVLogger(str(OUTPUT / "training_log.csv"), append=True),
    keras.callbacks.EarlyStopping(monitor="val_top1", mode="max", patience=10,
                                  restore_best_weights=True),
    keras.callbacks.TerminateOnNaN(),
]

resumed_epoch = 0
model = None
if RESUME and LAST_PATH.exists():
    print(f"Resuming from {LAST_PATH}")
    with strategy.scope():
        model = keras.models.load_model(LAST_PATH, compile=False)
    log_path = OUTPUT / "training_log.json"
    if log_path.exists():
        existing = json.loads(log_path.read_text())
        done = sum(len(existing.get(p, {}).get("loss", [])) for p in ("phase1", "phase2"))
        resumed_epoch = done
        print(f"  resumed at epoch {resumed_epoch}")

In [ ]:
hist1 = None
if resumed_epoch < PHASE1_EPOCHS:
    with strategy.scope():
        if model is None:
            model = build_model(NUM_CLASSES)
        compile_model(model, lr_schedule=LR_PHASE1)
    model.summary(line_length=110)

    print(f"\n=== Phase 1: head-only warmup (epochs {resumed_epoch} → {PHASE1_EPOCHS}) ===")
    hist1 = model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=resumed_epoch,
        epochs=PHASE1_EPOCHS,
        callbacks=callbacks,
        class_weight=class_weight,
    )
    resumed_epoch = PHASE1_EPOCHS

In [ ]:
hist2 = None
if resumed_epoch < EPOCHS:
    print(f"\n=== Phase 2: fine-tune top {FINE_TUNE_LAYERS} backbone layers "
          f"(epochs {resumed_epoch} → {EPOCHS}) ===")

    with strategy.scope():
        backbone = next(l for l in model.layers if isinstance(l, keras.Model))
        backbone.trainable = True
        for layer in backbone.layers[:-FINE_TUNE_LAYERS]:
            layer.trainable = False

        p2_epochs = EPOCHS - PHASE1_EPOCHS
        lr_sched = cosine_warmup_schedule(
            peak_lr=LR_PHASE2_PEAK,
            total_steps=p2_epochs * steps_per_epoch,
            warmup_steps=WARMUP_EPOCHS * steps_per_epoch,
        )
        compile_model(model, lr_schedule=lr_sched)

    hist2 = model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=resumed_epoch,
        epochs=EPOCHS,
        callbacks=callbacks,
        class_weight=class_weight,
    )

In [ ]:
log_path = OUTPUT / "training_log.json"
log = json.loads(log_path.read_text()) if log_path.exists() else {}

if hist1 is not None:
    log.setdefault("phase1", {})
    for k, v in hist1.history.items():
        log["phase1"].setdefault(k, []).extend(float(x) for x in v)
if hist2 is not None:
    log.setdefault("phase2", {})
    for k, v in hist2.history.items():
        log["phase2"].setdefault(k, []).extend(float(x) for x in v)

log_path.write_text(json.dumps(log, indent=2))
print("Done.")
print("Best:", BEST_PATH)
print("Last:", LAST_PATH)

In [ ]:
import numpy as np

model.load_weights(str(BEST_PATH))

def rep_gen():
    count = 0
    for batch, _ in train_ds.unbatch().batch(1):
        yield [tf.cast(batch, tf.float32).numpy()]
        count += 1
        if count >= 300:
            break

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = rep_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.float32
converter.inference_output_type = tf.float32

tflite_bytes = converter.convert()
tflite_path = OUTPUT / "flora_flower_classifier.tflite"
tflite_path.write_bytes(tflite_bytes)
print("TFLite:", tflite_path, f"({tflite_path.stat().st_size/1e6:.2f} MB)")